# Collapsi — Minimax con Alpha-Beta

Implementación correcta del juego **Collapsi** con IA basada en Minimax + Alpha-Beta + Deepening Iterativo.

**Reglas:** Cada turno debes moverte exactamente N pasos (N = valor de tu casilla de origen). La casilla de origen colapsa al terminar el turno. Gana el último jugador en completar su turno.

**Contrincantes disponibles:** `random`, `greedy`, `worst`, `ai` (IA vs IA), y humano interactivo.

In [63]:
import random
import time
import copy
import math
import pandas as pd

BOARD_SIZE = 4

WEIGHTS_CONFIG_1 = [3, 2, 2, -1, 1]
WEIGHTS_CONFIG_2 = [5, 1, 3, -2, 2]

In [64]:
# ==================================================
# GAME STATE - COLLAPSI (LÓGICA CORRECTA)
# ==================================================

class GameState:
    def __init__(self, board=None, positions=None, collapsed=None,
                 current_player=0, scores=None, moves_made=None):
        if board is not None:
            self.board = board
        else:
            # Tablero con valores 1-4 (cuántos pasos debe dar desde esa casilla)
            self.board = [[random.randint(1, 4) for _ in range(BOARD_SIZE)]
                          for _ in range(BOARD_SIZE)]

        # Posiciones iniciales: jugador 0 en esquina sup-izq, jugador 1 en esquina inf-der
        if positions is not None:
            self.positions = positions
        else:
            self.positions = [(0, 0), (BOARD_SIZE - 1, BOARD_SIZE - 1)]

        # Conjunto de casillas colapsadas (ya no disponibles)
        self.collapsed = collapsed if collapsed is not None else set()

        self.current_player = current_player
        self.scores = scores if scores is not None else [0, 0]
        # Cuántos turnos ha completado cada jugador
        self.moves_made = moves_made if moves_made is not None else [0, 0]

    def clone(self):
        return GameState(
            board=copy.deepcopy(self.board),
            positions=list(self.positions),
            collapsed=set(self.collapsed),
            current_player=self.current_player,
            scores=self.scores.copy(),
            moves_made=self.moves_made.copy()
        )

    def _wrap(self, r, c):
        """Aplica wrap-around estilo Pac-Man."""
        return r % BOARD_SIZE, c % BOARD_SIZE

    def _is_available(self, r, c):
        """Casilla disponible: no colapsada."""
        return (r, c) not in self.collapsed

    def _get_neighbors(self, r, c):
        """Vecinos válidos (arriba, abajo, izq, der) con wrap-around,
        excluyendo casillas colapsadas."""
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        neighbors = []
        for dr, dc in directions:
            nr, nc = self._wrap(r + dr, c + dc)
            if self._is_available(nr, nc):
                neighbors.append((nr, nc))
        return neighbors

    def get_moves(self):
        """
        Genera todos los caminos válidos para el jugador actual.
        Cada camino es una lista de posiciones [(r0,c0), (r1,c1), ...]
        donde r0,c0 es la posición actual y el camino tiene exactamente
        `steps` pasos (longitud steps+1).
        No se puede revisitar una casilla en el mismo turno.
        No se puede terminar en la casilla del oponente.
        """
        cp = self.current_player
        start = self.positions[cp]
        opponent_pos = self.positions[1 - cp]

        steps = self.board[start[0]][start[1]]

        paths = []
        self._dfs_paths(start, steps, [start], set([start]),
                        opponent_pos, paths)
        return paths

    def _dfs_paths(self, pos, remaining, path, visited,
                   opponent_pos, results):
        """DFS para encontrar todos los caminos de `remaining` pasos."""
        if remaining == 0:
            # El destino final no puede ser la casilla del oponente
            if pos != opponent_pos:
                results.append(list(path))
            return

        for nr, nc in self._get_neighbors(pos[0], pos[1]):
            if (nr, nc) not in visited:
                visited.add((nr, nc))
                path.append((nr, nc))
                self._dfs_paths((nr, nc), remaining - 1, path,
                                visited, opponent_pos, results)
                path.pop()
                visited.remove((nr, nc))

    def apply_move(self, path):
        """
        Aplica un movimiento:
        - La casilla de ORIGEN colapsa
        - El jugador se mueve a la posición final del camino
        - Suma puntos: valor de la casilla de origen
        """
        new_state = self.clone()

        cp = self.current_player
        origin = path[0]
        destination = path[-1]

        # La casilla de origen colapsa
        new_state.collapsed.add(origin)

        # Sumar puntos: valor de la casilla de origen
        new_state.scores[cp] += new_state.board[origin[0]][origin[1]]

        # Mover al jugador a la posición final
        new_state.positions[cp] = destination
        new_state.moves_made[cp] += 1

        # Cambiar turno
        new_state.current_player = 1 - cp

        return new_state

    def is_terminal(self):
        """
        El juego termina cuando el jugador actual NO puede
        completar ningún camino válido de los pasos requeridos.
        """
        return len(self.get_moves()) == 0

    def get_winner(self):
        """
        Gana el ÚLTIMO jugador que pudo completar su turno,
        es decir, el oponente del jugador que no puede moverse.
        """
        if self.is_terminal():
            return 1 - self.current_player
        return None

In [65]:
# ==================================================
# HEURÍSTICAS
# ==================================================


# Cada heurística se divide por su máximo para llevarla al rango [-1, 1]
_TOTAL_TILES = BOARD_SIZE * BOARD_SIZE   # 16
_MAX_VAL = 4

H1_MAX = _TOTAL_TILES                      # 16  — diferencia de turnos completados
H2_MAX = 108                               # 108 — caminos posibles
H3_MAX = BOARD_SIZE                       # 4   — vecinos accesibles del jugador
H4_MAX = BOARD_SIZE                       # 4   — vecinos accesibles del oponente
H5_MAX = (_TOTAL_TILES // 2) * _MAX_VAL   # 32  — diferencia de puntaje acumulado

_H_MAXS = [H1_MAX, H2_MAX, H3_MAX, H4_MAX, H5_MAX]


def evaluate(state, player, weights, num_heuristics):
    opponent = 1 - player

    h = []

    # H1: Diferencia de movimientos completados  →  rango bruto [-16, 16]
    raw_h1 = state.moves_made[player] - state.moves_made[opponent]
    h.append(raw_h1 / H1_MAX)

    # H2: Cantidad de movimientos disponibles    →  rango bruto [-108, 108]
    if state.current_player == player:
        raw_h2 = len(state.get_moves())
    else:
        raw_h2 = -len(state.get_moves())
    h.append(raw_h2 / H2_MAX)

    # H3: Vecinos accesibles del jugador         →  rango bruto [0, 4]
    pos = state.positions[player]
    raw_h3 = len(state._get_neighbors(pos[0], pos[1]))
    h.append(raw_h3 / H3_MAX)

    # H4: −Vecinos accesibles del oponente       →  rango bruto [-4, 0]
    opp_pos = state.positions[opponent]
    raw_h4 = len(state._get_neighbors(opp_pos[0], opp_pos[1]))
    h.append(-raw_h4 / H4_MAX)

    # H5: Diferencia de puntaje acumulado        →  rango bruto [-32, 32]
    raw_h5 = state.scores[player] - state.scores[opponent]
    h.append(raw_h5 / H5_MAX)

    total = 0
    for i in range(num_heuristics):
        total += weights[i] * h[i]

    return total

In [66]:
# ==================================================
# MINIMAX + ALPHA-BETA
# ==================================================

def minimax(state, depth, alpha, beta, maximizing,
            player, start_time, time_limit,
            stats, weights, num_heuristics):

    if time.time() - start_time >= time_limit:
        raise TimeoutError

    stats["nodes"] += 1

    if depth == 0 or state.is_terminal():
        return evaluate(state, player, weights, num_heuristics)

    moves = state.get_moves()

    if maximizing:
        value = -math.inf
        for move in moves:
            child = state.apply_move(move)
            value = max(value, minimax(child, depth - 1,
                                       alpha, beta, False,
                                       player, start_time,
                                       time_limit, stats,
                                       weights, num_heuristics))
            alpha = max(alpha, value)
            if beta <= alpha:
                break
        return value
    else:
        value = math.inf
        for move in moves:
            child = state.apply_move(move)
            value = min(value, minimax(child, depth - 1,
                                       alpha, beta, True,
                                       player, start_time,
                                       time_limit, stats,
                                       weights, num_heuristics))
            beta = min(beta, value)
            if beta <= alpha:
                break
        return value


def iterative_deepening(state, player, time_limit,
                        weights, num_heuristics):

    start_time = time.time()
    best_move = None
    depth = 1
    stats = {"nodes": 0, "depth": 0}

    # Fallback: primer movimiento disponible
    available = state.get_moves()
    if available:
        best_move = available[0]

    while True:
        try:
            best_value = -math.inf
            current_best = None

            for move in state.get_moves():
                child = state.apply_move(move)
                value = minimax(child, depth - 1,
                                -math.inf, math.inf,
                                False, player,
                                start_time, time_limit,
                                stats, weights,
                                num_heuristics)
                if value > best_value:
                    best_value = value
                    current_best = move

            if current_best is not None:
                best_move = current_best
            stats["depth"] = depth
            depth += 1

        except TimeoutError:
            break

    stats["time"] = time.time() - start_time
    return best_move, stats

In [67]:
# ==================================================
# JUGADORES
# ==================================================

def random_player(state):
    return random.choice(state.get_moves())

def greedy_player(state):
    """Elige el camino que termina con más vecinos disponibles."""
    moves = state.get_moves()
    def score(path):
        dest = path[-1]
        return len(state._get_neighbors(dest[0], dest[1]))
    return max(moves, key=score)

def worst_player(state):
    moves = state.get_moves()
    def score(path):
        dest = path[-1]
        return len(state._get_neighbors(dest[0], dest[1]))
    return min(moves, key=score)

In [68]:
# ==================================================
# EJECUTAR PARTIDA
# ==================================================

def run_game(ai_config, opponent_type, opponent_ai_config=None, verbose=False):
    """
    Ejecuta una partida completa.
    - ai_config:          configuración de la IA principal (jugador 0)
    - opponent_type:      "random" | "greedy" | "worst" | "ai"
    - opponent_ai_config: configuración de la segunda IA (solo si opponent_type="ai").
                          Si es None y opponent_type="ai", usa la misma config que J0.
    """
    if opponent_type == "ai" and opponent_ai_config is None:
        opponent_ai_config = ai_config

    state = GameState()
    total_stats    = {"nodes": 0, "depth": 0, "time": 0}
    opponent_stats = {"nodes": 0, "depth": 0, "time": 0}

    while not state.is_terminal():

        current = state.current_player

        if current == 0:
            move, stats = iterative_deepening(
                state, 0,
                ai_config["time"],
                ai_config["weights"],
                ai_config["heuristics"]
            )
            total_stats["nodes"] += stats["nodes"]
            total_stats["depth"]  = max(total_stats["depth"], stats["depth"])
            total_stats["time"]  += stats["time"]

        else:
            if opponent_type == "random":
                move = random_player(state)
            elif opponent_type == "greedy":
                move = greedy_player(state)
            elif opponent_type == "worst":
                move = worst_player(state)
            else:
                move, o_stats = iterative_deepening(
                    state, 1,
                    opponent_ai_config["time"],
                    opponent_ai_config["weights"],
                    opponent_ai_config["heuristics"]
                )
                opponent_stats["nodes"] += o_stats["nodes"]
                opponent_stats["depth"]  = max(opponent_stats["depth"], o_stats["depth"])
                opponent_stats["time"]  += o_stats["time"]

        if move is None:
            break

        state = state.apply_move(move)

    winner = state.get_winner()
    if winner is None:
        winner = 0 if state.scores[0] >= state.scores[1] else 1

    return winner, state.scores, total_stats, opponent_stats

In [69]:
# ==================================================
# PRINT BOARD
# ==================================================

def print_board(state):
    print("\nTABLERO:")
    p0 = state.positions[0]
    p1 = state.positions[1]
    for r in range(BOARD_SIZE):
        row_str = ""
        for c in range(BOARD_SIZE):
            if (r, c) in state.collapsed:
                cell = " X"
            else:
                val = str(state.board[r][c])
                if (r, c) == p0 and (r, c) == p1:
                    val = "B"   # ambos en misma casilla (no debería pasar al final)
                elif (r, c) == p0:
                    val = f"A{val}"  # jugador 0
                elif (r, c) == p1:
                    val = f"B{val}"  # jugador 1
                else:
                    val = f" {val}"
                cell = val
            row_str += cell.rjust(3)
        print(row_str)
    print(f"Posiciones: J0={state.positions[0]}  J1={state.positions[1]}")
    print(f"Colapsadas: {len(state.collapsed)} casillas")
    print(f"Puntajes: J0={state.scores[0]}  J1={state.scores[1]}")
    print()

In [70]:
# ==================================================
# HUMANO VS IA
# ==================================================

def get_reachable_next_steps(state, current_pos, steps_done, total_steps, visited):
    """
    Dado que el jugador ya recorrió `steps_done` pasos y está en `current_pos`,
    devuelve las casillas a las que puede moverse en el siguiente paso,
    SOLO si desde ahí existe al menos un camino completo para los pasos restantes.
    """
    opponent_pos = state.positions[1 - state.current_player]
    steps_left = total_steps - steps_done
    candidates = state._get_neighbors(current_pos[0], current_pos[1])
    valid = []
    for nxt in candidates:
        if nxt in visited:
            continue
        # Si es el último paso, no puede terminar en el oponente
        if steps_left == 1 and nxt == opponent_pos:
            continue
        # Si quedan más pasos, verificar que existe al menos un camino completo desde nxt
        if steps_left > 1:
            new_visited = visited | {nxt}
            if not _has_valid_path(state, nxt, steps_left - 1, new_visited, opponent_pos):
                continue
        valid.append(nxt)
    return valid


def _has_valid_path(state, pos, remaining, visited, opponent_pos):
    """Verifica si existe al menos un camino completo de `remaining` pasos desde pos."""
    if remaining == 0:
        return pos != opponent_pos
    for nxt in state._get_neighbors(pos[0], pos[1]):
        if nxt not in visited:
            if _has_valid_path(state, nxt, remaining - 1, visited | {nxt}, opponent_pos):
                return True
    return False


def human_vs_ai():

    print("=== COLLAPSI: HUMANO VS IA ===")
    print("Reglas: muévete exactamente N pasos (N = valor de tu casilla actual).")
    print("La casilla donde empiezas cada turno COLAPSA (X).")
    print("Gana el último en completar su turno.")
    print("Durante tu turno elige el SIGUIENTE paso a dar (no el camino completo).\n")

    time_limit = int(input("Tiempo máximo IA (segundos): "))
    h_count    = int(input("Cantidad de heurísticas (1-5): "))
    weight_option = int(input("Configuración de pesos (1 o 2): "))
    weights = WEIGHTS_CONFIG_1 if weight_option == 1 else WEIGHTS_CONFIG_2

    state = GameState()
    human_player = int(input("¿Quieres ser jugador 0 o 1?: "))

    while not state.is_terminal():

        print_board(state)
        current  = state.current_player
        origin   = state.positions[current]
        total_steps = state.board[origin[0]][origin[1]]
        print(f"Turno J{current} | Origen: {origin} | Pasos requeridos: {total_steps}")

        if current == human_player:
            # El humano elige paso a paso
            current_pos = origin
            visited     = {origin}
            path        = [origin]

            for step_num in range(1, total_steps + 1):
                steps_done = step_num - 1
                options = get_reachable_next_steps(
                    state, current_pos, steps_done, total_steps, visited
                )

                if not options:
                    print("⚠ Sin movimientos válidos desde aquí. Turno perdido.")
                    break

                print(f"\n  Paso {step_num}/{total_steps} — estás en {current_pos}")
                dirs = {
                    (current_pos[0] - 1) % BOARD_SIZE: "↑ Arriba",
                    (current_pos[0] + 1) % BOARD_SIZE: "↓ Abajo",
                }
                for i, pos in enumerate(options):
                    # Etiqueta de dirección
                    dr = pos[0] - current_pos[0]
                    dc = pos[1] - current_pos[1]
                    # Ajustar para wrap-around
                    if dr > BOARD_SIZE // 2:  dr -= BOARD_SIZE
                    if dr < -BOARD_SIZE // 2: dr += BOARD_SIZE
                    if dc > BOARD_SIZE // 2:  dc -= BOARD_SIZE
                    if dc < -BOARD_SIZE // 2: dc += BOARD_SIZE
                    if   dr == -1: direccion = "↑ Arriba"
                    elif dr ==  1: direccion = "↓ Abajo"
                    elif dc == -1: direccion = "← Izquierda"
                    else:          direccion = "→ Derecha"
                    print(f"    {i} -> {pos}  {direccion}")

                while True:
                    try:
                        idx = int(input(f"  Elige paso {step_num}: "))
                        if 0 <= idx < len(options):
                            break
                        print(f"  Ingresa un número entre 0 y {len(options)-1}")
                    except ValueError:
                        print("  Número inválido.")

                current_pos = options[idx]
                visited.add(current_pos)
                path.append(current_pos)

            move = path

        else:
            print("Turno IA pensando...")
            move, stats = iterative_deepening(
                state, current, time_limit, weights, h_count
            )
            # Mostrar los pasos que da la IA uno por uno
            print(f"IA se mueve: ", end="")
            for i, pos in enumerate(move[1:], 1):
                dr = pos[0] - move[i-1][0]
                dc = pos[1] - move[i-1][1]
                if dr > BOARD_SIZE//2:  dr -= BOARD_SIZE
                if dr < -BOARD_SIZE//2: dr += BOARD_SIZE
                if dc > BOARD_SIZE//2:  dc -= BOARD_SIZE
                if dc < -BOARD_SIZE//2: dc += BOARD_SIZE
                if   dr == -1: d = "↑"
                elif dr ==  1: d = "↓"
                elif dc == -1: d = "←"
                else:          d = "→"
                print(f"{d}{pos}", end="  ")
            print()
            print(f"Nodos: {stats['nodes']} | Prof: {stats['depth']} | T: {round(stats['time'],2)}s")

        state = state.apply_move(move)

    print_board(state)
    winner = state.get_winner()
    print(f"\n🏆 ¡Gana el jugador {winner}!")

In [71]:
# ==================================================
# BENCHMARK
# ==================================================

def benchmark():

    opponents = ["random", "greedy", "worst", "ai"]
    times = [1, 3, 10]
    heuristics_counts = [2, 3, 5]
    weight_configs = [WEIGHTS_CONFIG_1, WEIGHTS_CONFIG_2]

    wins = {opp: 0 for opp in opponents}
    total = {opp: 0 for opp in opponents}

    results_data = []

    for opponent in opponents:
        print(f"\n{'='*30}")
        print(f"CONTRINCANTE: {opponent}")
        print('='*30)

        for weights in weight_configs:
            for h_count in heuristics_counts:
                for t in times:

                    ai_config = {
                        "weights": weights,
                        "heuristics": h_count,
                        "time": t
                    }

                    winner, scores, stats, opp_stats = run_game(
                        ai_config, opponent,
                        opponent_ai_config=ai_config if opponent == "ai" else None
                    )
                    total[opponent] += 1
                    if winner == 0:
                        wins[opponent] += 1

                    print(f"Pesos:{weights} | H:{h_count} | T:{t}s")
                    print(f"  Ganador: J{winner} | Puntaje: {scores}")
                    print(f"  J0 — Nodos:{stats['nodes']} | Prof:{stats['depth']} | T:{round(stats['time'],2)}s")
                    if opponent == "ai":
                        print(f"  J1 — Nodos:{opp_stats['nodes']} | Prof:{opp_stats['depth']} | T:{round(opp_stats['time'],2)}s")
                    print("-"*40)

                    result_row = {
                        'Weights': str(weights),  # Convertir a string para mejor visualización
                        'Heuristics': h_count,
                        'Time_Limit': t,
                        'Opponent': opponent,
                        'Winner': winner,
                        'AI_Score': scores[0] if scores else 0,
                        'Opponent_Score': scores[1] if len(scores) > 1 else 0,
                        'Nodes_Explored': stats['nodes'],
                        'Max_Depth': stats['depth'],
                        'Game_Time': round(stats['time'], 2)
                    }
                    results_data.append(result_row)

    # Resumen en consola
    print("\n===== RESUMEN =====")
    for opp in opponents:
        w = wins[opp]
        t = total[opp]
        win_rate = round(100 * w / t if t else 0, 1)
        print(f"vs {opp}: {w}/{t} victorias ({win_rate}%)")

        # Agregar resumen al final del dataframe
        results_data.append({
            'Opponent': f'RESUMEN_{opp}',
            'Weights': 'SUMMARY',
            'Heuristics': '',
            'Time_Limit': '',
            'Winner': '',
            'AI_Score': win_rate,
            'Opponent_Score': w,
            'Nodes_Explored': t,
            'Max_Depth': '',
            'Game_Time': ''
        })

    # Crear DataFrame y guardar en Excel
    df = pd.DataFrame(results_data)
    df.to_excel('Collapsi_benchmark_results.xlsx', index=False)

In [72]:
# ==================================================
# MAIN
# ==================================================

if __name__ == "__main__":
    print("1 - Ejecutar Benchmark Automático")
    print("2 - Jugar Humano vs IA")
    print("3 - Partida rápida de prueba (IA vs random)")

    option = int(input("Seleccione opción: "))

    if option == 1:
        benchmark()
    elif option == 2:
        human_vs_ai()
    else:
        print("\nPartida de prueba: IA vs Random")
        ai_config = {"weights": WEIGHTS_CONFIG_1, "heuristics": 3, "time": 2}
        winner, scores, stats, _ = run_game(ai_config, "random")
        print(f"Ganador: {winner} | Puntajes: {scores}")
        print(f"Nodos: {stats['nodes']} | Profundidad: {stats['depth']} | Tiempo: {round(stats['time'],2)}s")

1 - Ejecutar Benchmark Automático
2 - Jugar Humano vs IA
3 - Partida rápida de prueba (IA vs random)
Seleccione opción: 2
=== COLLAPSI: HUMANO VS IA ===
Reglas: muévete exactamente N pasos (N = valor de tu casilla actual).
La casilla donde empiezas cada turno COLAPSA (X).
Gana el último en completar su turno.
Durante tu turno elige el SIGUIENTE paso a dar (no el camino completo).

Tiempo máximo IA (segundos): 3
Cantidad de heurísticas (1-5): 5
Configuración de pesos (1 o 2): 2
¿Quieres ser jugador 0 o 1?: 1

TABLERO:
 A2  2  3  3
  3  3  3  3
  3  1  2  1
  4  4  3 B3
Posiciones: J0=(0, 0)  J1=(3, 3)
Colapsadas: 0 casillas
Puntajes: J0=0  J1=0

Turno J0 | Origen: (0, 0) | Pasos requeridos: 2
Turno IA pensando...
IA se mueve: ↑(3, 0)  →(3, 1)  
Nodos: 56623 | Prof: 4 | T: 3.0s

TABLERO:
  X  2  3  3
  3  3  3  3
  3  1  2  1
  4 A4  3 B3
Posiciones: J0=(3, 1)  J1=(3, 3)
Colapsadas: 1 casillas
Puntajes: J0=2  J1=0

Turno J1 | Origen: (3, 3) | Pasos requeridos: 3

  Paso 1/3 — estás en (3